# SDS — Generate Dataset

Generate synthetic documents from registered templates.

**Setup** (run once):
```
pip install faker jinja2 playwright PyMuPDF rdflib pyyaml Pillow pandas numpy
playwright install chromium
```

## 1. Configuration

In [ ]:
import sys
from pathlib import Path

PROJECT = Path(".").resolve()
if not (PROJECT / "synthdoc").exists():
    PROJECT = PROJECT.parent
DATA_DIR = PROJECT / "data"

sys.path.insert(0, str(PROJECT))

from synthdoc.templates.registry import registry

templates = registry.list_templates()
print(f"{len(templates)} templates available:\n")
for i, t in enumerate(sorted(templates), 1):
    print(f"  {i:2}. {t}")

In [ ]:
TEMPLATES = None  # None = all registered templates
# TEMPLATES = [
#     "restaurant_receipt", "invoice", "business_card", "resume",
#     "bank_statement", "credit_card_statement", "irs_form_1040", "w2_form",
#     "passport", "driver_license",
#     "work_order", "aia_g702_g703", "bill_of_materials", "purchase_order",
#     "blueprint_technical_drawing",
#     "cms1500", "ub04_hospital_claims", "explanation_of_benefits",
#     "medical_prescription", "lab_report",
#     "closing_disclosure", "insurance_policy", "commercial_rent_roll",
#     "pay_stub", "remittance_advice",
#     "legal_pleading", "commercial_invoice", "certificate_of_analysis",
#     "home_inspection_report", "academic_transcript",
# ]

COUNT_PER_TEMPLATE = 1
WORKERS = 16  # parallel template workers

## 2. Generate

In [ ]:
import subprocess

cmd = [
    sys.executable, str(PROJECT / "generate_dataset.py"),
    "--count", str(COUNT_PER_TEMPLATE),
    "--workers", str(WORKERS),
]
if TEMPLATES:
    cmd += ["--templates"] + TEMPLATES

result = subprocess.run(cmd, cwd=str(PROJECT))
if result.returncode != 0:
    raise RuntimeError(f"Generation failed with exit code {result.returncode}")

## 3. Summary

In [ ]:
from synthdoc.bench.helpers import load_ner_pages, FULL_ENTITY_TYPES

pages = load_ner_pages(DATA_DIR, entity_types=FULL_ENTITY_TYPES)

print(f"Loaded {len(pages)} pages")
print(f"  Documents: {len(set(p['doc_id'] for p in pages))}")
print(f"  Templates: {len(set(p['template_type'] for p in pages))}")